# 09_export_decode_examples

Xuất bảng ví dụ giải mã posit (dùng trong báo cáo) — sinh bằng bộ giải mã đã kiểm chứng.

## Mục tiêu và cách đọc

Đọc sign/regime/exponent/fraction và giá trị giải mã. Ví dụ dùng cho phần lý thuyết và báo cáo.

## Cách sử dụng

1. Chọn kernel Python có `ipykernel`, `numpy`, `torch`, `matplotlib`.
2. Giữ `RUN_EXPERIMENT = False` và chọn **Run All** để xem dữ liệu sẵn có.
3. Đổi thành `True` nếu muốn thực hiện lại; chạy tuần tự các ô từ đầu.
4. **Ctrl+S / Cmd+S** lưu toàn bộ output vào notebook. Báo cáo HTML và CSV được xuất tự động ở ô cuối.

**Lưu ý:** các hình B/C/D là file đã lưu; sau khi đổi kết quả, chạy notebook 08 để vẽ lại trước khi dùng hình trong bài. Thời điểm xuất báo cáo không phải thời điểm thực hiện thí nghiệm gốc.


## 1. Thiết lập và chọn chế độ

Cài môi trường một lần theo `HUONG_DAN_NOTEBOOK.md`. Mỗi lần chạy tạo thư mục riêng trong `results/notebook_runs/`. Khi chạy lại thí nghiệm, các kết quả chuẩn và checkpoint hiện có được sao lưu trước.

In [ ]:
from pathlib import Path
import os
import sys

# Tìm 02_demo dù kernel bắt đầu ở thư mục workspace hoặc thư mục notebook.
candidates = [Path.cwd(), *Path.cwd().parents]
DEMO_ROOT = next((base / suffix for base in candidates
                  for suffix in ("", "02_demo", "Gk_XLDL/02_demo")
                  if (base / suffix / "posit_lab/numformats.py").is_file()), None)
if DEMO_ROOT is None:
    raise FileNotFoundError("Mở notebook trong workspace chứa Gk_XLDL/02_demo.")
DEMO_ROOT = DEMO_ROOT.resolve()
os.chdir(DEMO_ROOT)
if str(DEMO_ROOT) not in sys.path:
    sys.path.insert(0, str(DEMO_ROOT))
from notebook_tools.reporting import NotebookRun

# False: xem số liệu sẵn có. True: chạy code thí nghiệm bên dưới.
RUN_EXPERIMENT = False
SCRIPT_ARGS = []
__file__ = str(DEMO_ROOT / '09_export_decode_examples.py')
__package__ = None
session = NotebookRun(DEMO_ROOT, '09_export_decode_examples.py', RUN_EXPERIMENT, SCRIPT_ARGS)
print("Thư mục lưu lần này:", session.out)

# Cell magic giữ code thẳng hàng; vẫn áp dụng chế độ chạy và lưu nhật ký.
from IPython.core.magic import register_cell_magic
import __future__

@register_cell_magic
def experiment(line, cell):
    if not RUN_EXPERIMENT:
        print("Đã bỏ qua tính toán; đặt RUN_EXPERIMENT = True để chạy ô này.")
        return
    with session.execute():
        code = compile(cell, str(session.source), "exec",
                       flags=__future__.annotations.compiler_flag)
        exec(code, get_ipython().user_ns)


## 2. fields

Mã chuyển nguyên từ `09_export_decode_examples.py`. Ô chỉ thực thi khi `RUN_EXPERIMENT = True`; giữ thứ tự để các biến và hàm được khởi tạo đúng.

In [ ]:
%%experiment
"""Xuất bảng ví dụ giải mã posit (dùng trong báo cáo) — sinh bằng bộ giải mã đã kiểm chứng."""
import json
import os
from fractions import Fraction

from posit_lab.numformats import decode_posit

HERE = os.path.dirname(os.path.abspath(__file__))


def fields(bits, n, es):
    """Tách các trường sign | regime | exponent | fraction của một mẫu bit posit."""
    s = f"{bits:0{n}b}"
    if bits == 0:
        return {"bits": s, "kind": "zero", "value": 0.0}
    if bits == 1 << (n - 1):
        return {"bits": s, "kind": "NaR", "value": None}
    sign = bits >> (n - 1)
    mag = ((-bits) & ((1 << n) - 1)) if sign else bits
    body = f"{mag:0{n}b}"[1:]
    r0 = body[0]
    m = 1
    while m < len(body) and body[m] == r0:
        m += 1
    k = m - 1 if r0 == "1" else -m
    reg = body[:min(m + 1, len(body))]
    rest = body[len(reg):]
    e_str = rest[:es]
    f_str = rest[es:]
    e = int(e_str.ljust(es, "0"), 2) if es else 0
    f = Fraction(int(f_str, 2), 1 << len(f_str)) if f_str else Fraction(0)
    return {"bits": s, "sign": sign, "regime": reg, "k": k, "exp_bits": e_str, "e": e,
            "frac_bits": f_str, "f": str(f), "value": decode_posit(bits, n, es),
            "twos_complement_body": body if sign else None}


examples = {"posit<8,1>": [fields(b, 8, 1) for b in
                           [0x00, 0x01, 0x10, 0x20, 0x30, 0x40, 0x48, 0x50, 0x55, 0x56, 0x60, 0x70, 0x7F,
                            0x80, 0xC0]],
            "posit<5,1>_all_positive": [fields(b, 5, 1) for b in range(0, 16)],
            "posit<16,1>_2.56": None}
# Ví dụ 2.56 của [01] (P(16,1)): tìm posit gần nhất
import torch
from posit_lab.numformats import quantize_posit
v = float(quantize_posit(torch.tensor([2.56], dtype=torch.float64), 16, 1)[0])


## 3. Chuẩn bị / tính toán / lưu dữ liệu theo thứ tự script

Mã chuyển nguyên từ `09_export_decode_examples.py`. Ô chỉ thực thi khi `RUN_EXPERIMENT = True`; giữ thứ tự để các biến và hàm được khởi tạo đúng.

In [ ]:
%%experiment
for b in range(1 << 15):
    if decode_posit(b, 16, 1) == v:
        examples["posit<16,1>_2.56"] = fields(b, 16, 1)
        break
with open(os.path.join(HERE, "results", "09_decode_examples.json"), "w") as f:
    json.dump(examples, f, indent=1, ensure_ascii=False)
for r in examples["posit<8,1>"]:
    print(r)
print(examples["posit<16,1>_2.56"])


## Kết quả, bảng số liệu và hình

Ô cuối đọc đầy đủ các trường JSON, xuất CSV UTF-8 cho Excel, lưu JSON gốc, nhúng hình vào HTML để mở độc lập. Các bảng dài có thanh cuộn. Khi viết bài, ghi rõ cấu hình, seed, tập dữ liệu và giới hạn mô phỏng.

In [ ]:
session.report(['09_decode_examples.json'], [])

bits,kind,value,sign,regime,k,exp_bits,e,frac_bits,f,twos_complement_body
00000000,zero,0.0,,,,,,,,
00000001,,0.000244140625,0,0000001,-6,,0,,0,None
00010000,,0.0625,0,001,-2,0,0,000,0,None
00100000,,0.25,0,01,-1,0,0,0000,0,None
00110000,,0.5,0,01,-1,1,1,0000,0,None
01000000,,1.0,0,10,0,0,0,0000,0,None
01001000,,1.5,0,10,0,0,0,1000,1/2,None
01010000,,2.0,0,10,0,1,1,0000,0,None
01010101,,2.625,0,10,0,1,1,0101,5/16,None
01010110,,2.75,0,10,0,1,1,0110,3/8,None
